In [ ]:
# This line installs the latest version of cvxpy, which is needed to use the
# IPOPT solver below. After the next cvxpy release, just importing cvxpy will be
# sufficient.
!pip install git+https://github.com/cvxpy/cvxpy.git

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# Problem data.
K = 100
m = 1
g = 10
l = 1
h = .05

# Initial and final state.
x0 = np.array([np.pi, 0])
xK = np.array([0, 0])

In [ ]:
# Pendulum dynamics in state-space form.
def f(x, u):
  return cp.hstack([
      x[1],
      (m * g * l * cp.sin(x[0]) + u) / (m * l ** 2)])

In [ ]:
# Variables.
x = cp.Variable((K + 1, 2))
u = cp.Variable(K)

# Constraints.
constraints = [x[0] == x0, x[-1] == xK]
for k in range(K):
  constraints += [x[k + 1] == x[k] + h * f(x[k], u[k])]

# Objective function.
obj = cp.sum_squares(u)

# Initial guess.
u.value = np.zeros(K)
x.value = np.linspace(x0, xK, K + 1)

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(nlp=True, solver=cp.IPOPT)

In [ ]:
# Plot state trajectory.
plt.figure()
plt.plot(x[:, 0].value, x[:, 1].value)
plt.xlabel(r'Angle $q$')
plt.ylabel(r'Angular velocity $\dot q$')
plt.title('Pendulum trajectory')
plt.grid()

In [ ]:
# Animate pendulum motion.
fig, ax = plt.subplots()
ax.set_xlim(-l - .2, l + .2)
ax.set_ylim(-l - .2, l + .2)
ax.set_aspect('equal')
ax.grid()
line, = ax.plot([], [], 'o-')
def init():
    line.set_data([], [])
    return line,

# Animation update function.
hor = l * np.sin(x.value[:, 0])
vert = l * np.cos(x.value[:, 0])
def update(frame):
    line.set_data([0, hor[frame]], [0, vert[frame]])
    return line,

# Create animation.
ani = FuncAnimation(fig, update, frames=K+1, init_func=init, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())